# import from Kaggle

In [ ]:
! pip install opendatasets

In [ ]:
import opendatasets as od

In [ ]:
od.download('https://www.kaggle.com/datasets/jolenechen/datacosupplychaindataset')


# export to MongoDb

In [ ]:
! pip install pymongo

In [ ]:
from pymongo import MongoClient


In [ ]:
client = MongoClient(os.getenv("MONGO_URI"))

db = client['Regression_loogistic']
collection = db['preprocessing']

inserted_document = collection.insert_one(document)

print(f"Inserted Document ID: {inserted_document.inserted_id}")
#client.close()

In [ ]:
data = df.to_dict("records")
data_dict = df.to_dict(orient='records')
collection.insert_many(data_dict)


# Pyspark Logistic Regression model

In [ ]:
from tqdm import tqdm

batch_size = 1000   # ممتاز لـ 100MB

for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i+batch_size]
    collection.insert_many(batch)

client.close()

In [ ]:
# Install Java Development Kit (required for Spark)
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Install Spark
!pip install pyspark

# Install the MongoDB Spark Connector package
# Use the correct Spark version (e.g., spark-3.0) and Scala version (e.g., scala-2.12)
!pip install findspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("Colab PySpark").getOrCreate()
from pyspark.ml.feature import StringIndexer
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
from pyspark.ml import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


In [ ]:
dc = spark.read.csv("DataCoSupplyChainDataset.csv", header=True, inferSchema=True)

In [ ]:
dsc = dc.drop("Customer Email", "Customer Country", "Customer Password", "Longitude", "Latitude", "Customer Street", "Product Image", "order date (DateOrders)", "Product Description", "Order Zipcode", "Shipping Zipcode")


# Data Cleaning

In [ ]:
from pyspark.sql.functions import col, sum
dsc.select([sum(col(c).isNull().cast("int")).alias(c) for c in dsc.columns]).show()
#Customer Zipcode-->3 , Customer Lname ---->5

In [ ]:
dc_c = dsc.dropna()

In [ ]:
numerical_columns = ['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Category Id',
                     'Customer Id', 'Customer Zipcode', 'Department Id', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id','Order Item Discount',
                     'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price','Order Item Profit Ratio', 'Order Item Quantity', 'Sales',
                     'Order Item Total', 'Product Card Id', 'Order Profit Per Order', 'Product Category Id', 'Product Price', 'Product Status'

                     ]


In [ ]:
categorical_cols = [
    "Type",
    "Delivery Status",
    "Category Name",
    "Customer City",
    "Customer Fname",
    "Customer Lname",
    "Customer Segment",
    "Customer State",
    "Department Name",
    "Market",
    "Order City",
    "Order Country",
    "Order Region",
    "Order State",
    "Order Status",
    "Product Name",
    "Shipping Mode"
]


# **Encoding**

In [ ]:
# index the string columns
indexers = [StringIndexer(inputCol=col, outputCol="{0}_index".format(col)) for col in categorical_cols]

# encode the indexed values
encoders = [OneHotEncoder(dropLast=False, inputCol=indexer.getOutputCol(), outputCol="{0}_encoded".format(indexer.getOutputCol()))
           for indexer in indexers]

input_columns = [encoder.getOutputCol() for encoder in encoders] + numerical_columns
assembler = VectorAssembler(inputCols=input_columns, outputCol="feature")

assemling Indexer for corelation *matrix*

In [ ]:
pipeline = Pipeline(stages=indexers)
df_encoded = pipeline.fit(dc_c).transform(dc_c)

In [ ]:
df_encoded.show()

In [ ]:
second_drop = df_encoded.drop("Type", "Delivery Status", "Category Name", "Customer City", "Customer Fname", "Customer Lname", "Customer Segment", "Customer State", "Department Name", "Market", "Order City", "Order Country", "Order Region", "Order State", "shipping date (DateOrders)", "Order Status", "Product Name", "Shipping Mode")


In [ ]:
second_drop.show()

# Correlation Matrix

In [ ]:
pandas_file = second_drop.toPandas()

In [ ]:
corr_matrix = pandas_file.corr().round(2)
plt.figure(figsize=(20,20))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')


In [ ]:
my_matrix = pandas_file.corr(method="spearman").round(2)
my_matrix.style.background_gradient()

In [ ]:
import numpy as np

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [ ]:
high_corr = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.8)
]

high_corr


In [ ]:
high_corr_pairs = (upper_triangle.stack().reset_index()
)

high_corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']

high_corr_pairs = high_corr_pairs[
    high_corr_pairs['Correlation'] > 0.85
]

high_corr_pairs


In [ ]:
to_drop = set(high_corr_pairs['Feature 2'])


In [ ]:
data_clean = pandas_file.drop(columns=list(to_drop))


In [ ]:
len(data_clean.columns)

In [ ]:

len(pandas_file.columns)

Correlation wint Dependent Variable

In [ ]:
target = "Late_delivery_risk"

corr_with_target = (data_clean.corr()[target].abs().sort_values(ascending=False)
)

corr_with_target


# PreProcessing Assempling

In [ ]:
pipeline2 = Pipeline(stages=indexers + encoders + [assembler])
df_final2 = pipeline2.fit(dc_c).transform(dc_c)


In [ ]:
pipeline2

In [ ]:
# pipeline = Pipeline(stages=indexers)
# df_encoded = pipeline.fit(dc_c).transform(dc_c)

In [ ]:
df_final2.show(10)

In [ ]:
training_data2 = df_final2.drop("Type",
                                "Delivery Status",
                                "Category Name",
                                "Customer City",
                                "Customer Fname",
                                "Customer Lname",
                                "Customer Segment",
                                "Customer State",
                                "Department Name",
                                "Market",
                                "Order City",
                                "Order Country",
                                "Order Region",
                                "Order State",
                                "shipping date (DateOrders)",
                                "Order Status",
                                "Product Name",
                                "Shipping Mode")


In [ ]:
training_data2.show(10)

# Training

In [ ]:
# training_data2.columns


In [ ]:
basic_features_1 = [
    "Delivery Status_index_encoded",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Shipping Mode_index_encoded"

]

In [ ]:
feature_cols_advanc = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Shipping Mode_index",
    "Order Status_index",
    "Type_index",
    "Customer Zipcode",
    "Customer City_index",
    "Order Item Quantity",
    "Department Id",
    "Customer Segment_index",
    "Customer Lname_index",
    "Customer State_index",
]


# Assembling Trainin Features




In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols_advanc,
    outputCol="advanced_features"
)

advanced_basic2 = assembler.transform(training_data2)
advanced_basic2.show()


In [ ]:

from pyspark.ml.feature import StandardScaler

scaler6 = StandardScaler(
    inputCol="advanced_features",
    outputCol="trained_advanced_features",
    withMean=True,
    withStd=True
)

all = scaler6.fit(advanced_basic2).transform(advanced_basic2)


In [ ]:
all.show()

In [ ]:
train_data, test_data = all.randomSplit([0.8, 0.2], seed=42)


In [ ]:
train_data.show()

In [ ]:
from pyspark.ml.classification import LogisticRegression

log_re = LogisticRegression(featuresCol="trained_advanced_features",labelCol="Late_delivery_risk")

logistic_model = log_re.fit(train_data)

# Evalutation

In [ ]:
test = logistic_model.transform(test_data)

test.select("Late_delivery_risk","probability","prediction").show(10)


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluating = BinaryClassificationEvaluator(labelCol="Late_delivery_risk")

aucss = evaluating.evaluate(test)
print("AUC:", aucss)


In [ ]:
test.groupBy("Late_delivery_risk", "prediction").count().show()


Recall

In [ ]:
recall = MulticlassClassificationEvaluator(
    labelCol="Late_delivery_risk",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(test)

print("Recall:", recall)


F1-score

In [ ]:
f1 = MulticlassClassificationEvaluator(
    labelCol="Late_delivery_risk",
    predictionCol="prediction",
    metricName="f1"
).evaluate(test)

print("F1-score:", f1)



